In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
!apt-get update -y && apt-get install -y ffmpeg

# (2) Make sure PyAV is installed & current (your log shows it's already 15.1.0)
!pip install --upgrade --no-cache-dir av


Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [3]:
!git clone https://github.com/Atze00/MoViNet-pytorch.git

fatal: destination path 'MoViNet-pytorch' already exists and is not an empty directory.


In [4]:
%cd MoViNet-pytorch
!pip install -r tests/test_requirements.txt
!pip install torch torchvision torchaudio opencv-python

/content/MoViNet-pytorch
  Cloning https://github.com/Atze00/MoViNet-pytorch.git to /tmp/pip-req-build-nsbh85tq
  Running command git clone --filter=blob:none --quiet https://github.com/Atze00/MoViNet-pytorch.git /tmp/pip-req-build-nsbh85tq
  Resolved https://github.com/Atze00/MoViNet-pytorch.git to commit c2d1edf48fc6c5259707f9d833f22171b4f63493
  Preparing metadata (setup.py) ... done


Prepare my dataset for movinet, I will split into 6 clips by 5 frames

In [5]:
import cv2
import numpy as np

def extract_6x5_clips(video_path, num_clips=6, frames_per_clip=5, resize=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < num_clips * frames_per_clip:
        cap.release()
        return None  # skip too short videos

    # Divide the video into 6 segments
    segment_length = total_frames // num_clips
    clips = []

    for c in range(num_clips):
        start = c * segment_length
        step = max(segment_length // frames_per_clip, 1)
        frames = []

        for f in range(frames_per_clip):
            frame_idx = start + f * step
            if frame_idx >= total_frames:
                break
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, resize)
            frames.append(frame)

        if len(frames) == frames_per_clip:
            clips.append(np.stack(frames))  # shape (5, H, W, 3)

    cap.release()
    return clips  # list of 6 arrays of shape (5, H, W, 3)


In [6]:
import os, glob, random, torch
from torch.utils.data import Dataset
import torchvision.io as io
import torchvision
import av
CLASS_TO_IDX = {"NonFight": 0, "Fight": 1}
VIDEO_EXTS = (".avi")

def list_videos(root_split_dir):  # e.g. root_split_dir="RWF2000/train"
    items = []
    for cls in CLASS_TO_IDX:
        for p in glob.glob(os.path.join(root_split_dir, cls, "**"), recursive=True):
            if os.path.isfile(p) and p.lower().endswith(VIDEO_EXTS):
                items.append((p, CLASS_TO_IDX[cls]))
    items.sort()
    return items

class RWFClipsDataset(Dataset):
    def __init__(self, root_dir, split="train", num_clips=6, frames_per_clip=5, transform=None, random_clip_order=False):
        self.items = list_videos(os.path.join(root_dir, split))
        self.num_clips = num_clips
        self.frames_per_clip = frames_per_clip
        self.transform = transform
        self.random_clip_order = random_clip_order

    def __len__(self):
        return len(self.items)  # one sample per video

    def _extract_6x5(self, video):  # video: (T, C, H, W)
        T = video.shape[0]
        k = self.num_clips
        f = self.frames_per_clip

        # guard for very short videos
        if T < k * f:
            # loop video frames to reach enough length
            reps = (k * f + T - 1) // T
            video = torch.cat([video] * reps, dim=0)
            T = video.shape[0]

        # split timeline into k equal sections and take f consecutive frames from each
        stride = T // k
        clips = []
        for i in range(k):
            start_max = max(0, stride * (i + 1) - f)  # ensure we can take f frames
            start_min = stride * i
            start = start_min if not self.random_clip_order else random.randint(start_min, start_max)
            clip = video[start:start + f]  # (f, C, H, W)
            if clip.shape[0] < f:  # pad last frames if any off-by-one
                pad = f - clip.shape[0]
                clip = torch.cat([clip, video[-pad:]], dim=0)
            if self.transform:
                clip = self.transform(clip)  # expect transform to handle (T, C, H, W)
            clips.append(clip)
        return torch.stack(clips, dim=0)  # (k, f, C, H, W)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        # read_video returns (T, H, W, C)
        frames, _, _ = io.read_video(path, pts_unit='sec')
        frames = frames.permute(0, 3, 1, 2).to(torch.uint8)  # (T, C, H, W)

        clips = self._extract_6x5(frames)
        return clips, label


In [7]:
import torch.nn.functional as F
import torchvision.transforms as T
import torch.optim as optim

# Train transform
transform = T.Compose([
    T.ConvertImageDtype(torch.float32),
    T.Resize((200, 200)),
    T.RandomHorizontalFlip(),
    T.RandomCrop((172, 172)),
])

# Test (validation) transform
transform_test = T.Compose([
    T.ConvertImageDtype(torch.float32),
    T.Resize((200, 200)),
    T.CenterCrop((172, 172)),
])


In [8]:
train_dataset = RWFClipsDataset("/content/drive/MyDrive/datasets/rwf2000", transform=transform)
val_dataset = RWFClipsDataset("/content/drive/MyDrive/datasets/rwf2000", split="val", transform=transform_test)

In [9]:
batch_size=16
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [10]:
from einops import rearrange
import torch.nn.functional as F

def forward_clips_then_aggregate(model, x):
    # x: [b, n, t, c, h, w]
    b, n, t, c, h, w = x.shape
    x5d = rearrange(x, 'b n t c h w -> (b n) c t h w')  # per-clip batches
    logits_per_clip = model(x5d)                         # [(b*n), num_classes]
    logits_per_clip = rearrange(logits_per_clip, '(b n) c -> b n c', b=b, n=n)
    # Aggregate clips to a single video logit (choose one):
    video_logits = logits_per_clip.mean(dim=1)           # simple mean
    # Or: probability-space mean (often nicer):
    # video_logits = torch.logsumexp(logits_per_clip, dim=1) - math.log(n)
    return video_logits, logits_per_clip

In [11]:
from einops import rearrange
import torch.nn.functional as F
import torch

def train_iter_stream(model, optimz, data_load, loss_val, n_clips=6, n_clip_frames=5):
    """
    Trains in causal mode, feeding subclips sequentially.
    Accepts data of shape:
      - [b, n_clips, n_clip_frames, c, h, w]  (preferred)
      - [b, c, t, h, w] with t == n_clips * n_clip_frames
    """
    samples = len(data_load.dataset)
    model.cuda()
    model.train()
    model.clean_activation_buffers()
    optimz.zero_grad()

    for i, (data, target) in enumerate(data_load):
        data = data.cuda(non_blocking=True)
        target = target.cuda(non_blocking=True)

        # We'll accumulate the batch-average loss across clips (already scaled by 1/n_clips)
        l_batch = 0.0

        if data.ndim == 6:
            # [b, n, t, c, h, w]
            b, n, t, c, h, w = data.shape
            assert n == n_clips and t == n_clip_frames, \
                f"Got data of shape {tuple(data.shape)}, but expected n_clips={n_clips}, n_clip_frames={n_clip_frames}"
            # Keep MoViNet states across clip calls; don't clean until after optimizer step
            for j in range(n):
                clip_5d = rearrange(data[:, j], 'b t c h w -> b c t h w').contiguous()
                logits = model(clip_5d)
                logp = F.log_softmax(logits, dim=1)
                loss = F.nll_loss(logp, target) / n  # average over clips
                loss.backward()
                l_batch += loss.item()
        elif data.ndim == 5:
            # [b, c, t, h, w]
            b, c, T, h, w = data.shape
            assert T == n_clips * n_clip_frames, \
                f"T={T} must equal n_clips*n_clip_frames={n_clips*n_clip_frames}"
            for j in range(n_clips):
                clip_5d = data[:, :, j*n_clip_frames:(j+1)*n_clip_frames, :, :]
                logits = model(clip_5d)
                logp = F.log_softmax(logits, dim=1)
                loss = F.nll_loss(logp, target) / n_clips
                loss.backward()
                l_batch += loss.item()
        else:
            raise ValueError(f"Unexpected data.ndim={data.ndim}; expected 5 or 6")

        optimz.step()
        optimz.zero_grad()
        model.clean_activation_buffers()

        if i % 2 == 0:
            print('[' +  '{:5}'.format(i * len(data)) + '/' + '{:5}'.format(samples) +
                  ' (' + '{:3.0f}'.format(100 * i / len(data_load)) + '%)]  Loss: ' +
                  '{:6.4f}'.format(l_batch))
            loss_val.append(l_batch)


def evaluate_stream(model, data_load, loss_val, n_clips=6, n_clip_frames=5):
    """
    Evaluation that aggregates **all clips** per video before computing loss & accuracy.
    Keeps MoViNet states across clips within a video; resets per video.
    """
    model.eval()
    model.cuda()
    samples = len(data_load.dataset)
    correct = 0
    total_loss = 0.0

    with torch.no_grad():
        for i, (data, target) in enumerate(data_load):
            data = data.cuda(non_blocking=True)
            target = target.cuda(non_blocking=True)

            model.clean_activation_buffers()  # reset states at video start

            # We'll accumulate log-probs across clips and average them
            accum_logp = None
            n_used = 0

            if data.ndim == 6:
                # [b, n, t, c, h, w]
                b, n, t, c, h, w = data.shape
                for j in range(n):
                    clip_5d = rearrange(data[:, j], 'b t c h w -> b c t h w').contiguous()
                    logits = model(clip_5d)
                    logp = F.log_softmax(logits, dim=1)
                    accum_logp = logp if accum_logp is None else (accum_logp + logp)
                    n_used += 1
                    n_used += 1
            else:
                raise ValueError(f"Unexpected data.ndim={data.ndim}; expected 6")

            avg_logp = accum_logp / n_used
            loss = F.nll_loss(avg_logp, target)
            pred = avg_logp.argmax(dim=1)

            total_loss += loss.item()
            correct += pred.eq(target).sum().item()

    aloss = total_loss / len(data_load)
    loss_val.append(aloss)
    print('\nAverage test loss: {:.4f}  Accuracy: {}/{} ({:.2f}%)\n'
          .format(aloss, correct, samples, 100.0 * correct / samples))


In [12]:
!pip uninstall -y av && pip install --no-cache-dir av


Found existing installation: av 15.1.0
Uninstalling av-15.1.0:
  Successfully uninstalled av-15.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 123.7 MB/s eta 0:00:00


In [13]:
import time
from movinets import MoViNet
from movinets.config import _C

N_EPOCHS = 1

model = MoViNet(_C.MODEL.MoViNetA0, causal = True, pretrained = True )
start_time = time.time()

trloss_val, tsloss_val = [], []
model.classifier[3] = torch.nn.Conv3d(2048, 2, (1,1,1))
optimz = optim.Adam(model.parameters(), lr=0.00005)
for epoch in range(1, N_EPOCHS + 1):
    print('Epoch:', epoch)
    train_iter_stream(model, optimz, train_loader, trloss_val)
    evaluate_stream(model, val_loader, tsloss_val)

print('Execution time:', '{:5.2f}'.format(time.time() - start_time), 'seconds')

Epoch: 1


/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


[    0/ 1600 (  0%)]  Loss: 0.7132
[   32/ 1600 (  2%)]  Loss: 0.7084
[   64/ 1600 (  4%)]  Loss: 0.7023
[   96/ 1600 (  6%)]  Loss: 0.6804
[  128/ 1600 (  8%)]  Loss: 0.6743
[  160/ 1600 ( 10%)]  Loss: 0.6879
[  192/ 1600 ( 12%)]  Loss: 0.7060
[  224/ 1600 ( 14%)]  Loss: 0.6532
[  256/ 1600 ( 16%)]  Loss: 0.6620
[  288/ 1600 ( 18%)]  Loss: 0.6842
[  320/ 1600 ( 20%)]  Loss: 0.6619
[  352/ 1600 ( 22%)]  Loss: 0.6347
[  384/ 1600 ( 24%)]  Loss: 0.5949
[  416/ 1600 ( 26%)]  Loss: 0.6271
[  448/ 1600 ( 28%)]  Loss: 0.6263
[  480/ 1600 ( 30%)]  Loss: 0.7239
[  512/ 1600 ( 32%)]  Loss: 0.6519
[  544/ 1600 ( 34%)]  Loss: 0.6148
[  576/ 1600 ( 36%)]  Loss: 0.6108
[  608/ 1600 ( 38%)]  Loss: 0.6184
[  640/ 1600 ( 40%)]  Loss: 0.6645
[  672/ 1600 ( 42%)]  Loss: 0.6959
[  704/ 1600 ( 44%)]  Loss: 0.6332
[  736/ 1600 ( 46%)]  Loss: 0.5764
[  768/ 1600 ( 48%)]  Loss: 0.5575
[  800/ 1600 ( 50%)]  Loss: 0.6240
[  832/ 1600 ( 52%)]  Loss: 0.5385
[  864/ 1600 ( 54%)]  Loss: 0.6667
[  896/ 1600 ( 56%)]